# Modelagem e Avaliação (v5)
## Modelo A (v4, 216 amostras) vs Modelo B (v5 híbrido, 96 amostras)

**Hipótese:** Informações de histórico de Copas e valor de mercado do elenco compensam a redução no número de amostras de treino.

| | Modelo A | Modelo B |
|--|---------|----------|
| Features | 9 (v4) | 12 (v5) |
| Treino | 1994–2018 (216) | 2010–2018 (96) |
| Teste | Copa 2022 (32) | Copa 2022 (32) |
| Valor de mercado | ❌ | ✅ |

## 1. Imports e Carregamento

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor
from scipy.stats import wilcoxon

sns.set_theme(style='whitegrid')
np.random.seed(42)

df_v4 = pd.read_csv('../data/processed/features_completo_v4.csv')
df_v5 = pd.read_csv('../data/processed/features_completo_v5.csv')

print(f'v4: {df_v4.shape} | v5: {df_v5.shape}')

## 2. Definição das Features e Divisão

**Modelo A:** treino em 1994–2018, features v4 (sem valor de mercado)

**Modelo B:** treino em 2010–2018 (onde valor de mercado existe), features v5 completas

In [ ]:
features_v4 = [
    'media_gols_marcados_ciclo', 'media_gols_sofridos_ciclo',
    'pct_vitorias_ciclo', 'total_jogos_ciclo',
    'media_gols_marcados_ult15', 'media_gols_sofridos_ult15',
    'pct_vitorias_ult15', 'elo_medio_adv_ciclo', 'elo_medio_adv_ult15'
]

features_v5 = features_v4 + [
    'media_gols_ultimas2_copas', 'fase_ultima_copa', 'valor_mercado_milhoes'
]

# Modelo A — v4 completo
X_tr_A = df_v4[df_v4['copa_alvo'] < 2022][features_v4]
X_te_A = df_v4[df_v4['copa_alvo'] == 2022][features_v4]
y_tr_A = df_v4[df_v4['copa_alvo'] < 2022]['media_gols_copa']
y_te   = df_v4[df_v4['copa_alvo'] == 2022]['media_gols_copa']

# Modelo B — v5 híbrido (só 2010–2018 no treino)
df_v5_treino = df_v5[
    (df_v5['copa_alvo'] >= 2010) &
    (df_v5['copa_alvo'] < 2022)
]
X_tr_B = df_v5_treino[features_v5]
X_te_B = df_v5[df_v5['copa_alvo'] == 2022][features_v5]
y_tr_B = df_v5_treino['media_gols_copa']

# Imputar NaN no valor de mercado (mediana do treino)
imputer = SimpleImputer(strategy='median')
X_tr_B_imp = imputer.fit_transform(X_tr_B)
X_te_B_imp = imputer.transform(X_te_B)

print(f'Modelo A — Treino: {X_tr_A.shape[0]} | Teste: {X_te_A.shape[0]}')
print(f'Modelo B — Treino: {X_tr_B.shape[0]} | Teste: {X_te_B.shape[0]}')

## 3. Treino e Avaliação dos Modelos

In [ ]:
resultados = []
modelos_A  = {}
modelos_B  = {}

configs = [
    ('Regressão Linear', LinearRegression()),
    ('Random Forest',    RandomForestRegressor(n_estimators=100, random_state=42)),
    ('XGBoost',          XGBRegressor(n_estimators=100, random_state=42))
]

for nome, _ in configs:
    # Modelo A
    from sklearn.base import clone
    m_a = clone(_)
    m_a.fit(X_tr_A, y_tr_A)
    pred_a = m_a.predict(X_te_A)
    modelos_A[nome] = m_a
    resultados.append({
        'Modelo': nome, 'Versão': 'A — v4 (216 amostras)',
        'MAE': mean_absolute_error(y_te, pred_a),
        'RMSE': root_mean_squared_error(y_te, pred_a)
    })

    # Modelo B
    m_b = clone(_)
    m_b.fit(X_tr_B_imp, y_tr_B)
    pred_b = m_b.predict(X_te_B_imp)
    modelos_B[nome] = m_b
    resultados.append({
        'Modelo': nome, 'Versão': 'B — v5 (96 amostras)',
        'MAE': mean_absolute_error(y_te, pred_b),
        'RMSE': root_mean_squared_error(y_te, pred_b)
    })

df_res = pd.DataFrame(resultados)
print('Comparação Modelo A vs Modelo B — Teste (Copa 2022):')
print(df_res.sort_values(['Modelo', 'Versão']).to_string(index=False))

## 4. Visualização — MAE A vs B

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, metrica in zip(axes, ['MAE', 'RMSE']):
    pivot = df_res.pivot(index='Modelo', columns='Versão', values=metrica)
    pivot.plot(kind='bar', ax=ax, color=['steelblue', 'seagreen'],
               edgecolor='black', alpha=0.85)
    ax.set_title(f'{metrica} — Modelo A vs B')
    ax.set_ylabel(metrica)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('Modelo A (v4, 216 amostras) vs Modelo B (v5 híbrido, 96 amostras)\nCopa 2022', fontsize=12)
plt.tight_layout()
plt.savefig('../article/figures/comparacao_modeloA_modeloB.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Validação Cruzada — Modelo B

In [ ]:
copas_B = sorted(df_v5[
    (df_v5['copa_alvo'] >= 2010) &
    (df_v5['copa_alvo'] <= 2022)
]['copa_alvo'].unique())

folds_B = [(copas_B[:i], copas_B[i]) for i in range(1, len(copas_B))]

mae_cv_A = {'LR': [], 'RF': [], 'XGB': []}
mae_cv_B = {'LR': [], 'RF': [], 'XGB': []}

for copas_tr, copa_te in folds_B:
    # Modelo A
    mask_tr = df_v4['copa_alvo'].isin(copas_tr)
    mask_te = df_v4['copa_alvo'] == copa_te
    Xtr_a = df_v4[mask_tr][features_v4]
    ytr_a = df_v4[mask_tr]['media_gols_copa']
    Xte_a = df_v4[mask_te][features_v4]
    yte   = df_v4[mask_te]['media_gols_copa']

    # Modelo B
    mask_tr_b = df_v5['copa_alvo'].isin(copas_tr)
    mask_te_b = df_v5['copa_alvo'] == copa_te
    Xtr_b_raw = df_v5[mask_tr_b][features_v5]
    ytr_b     = df_v5[mask_tr_b]['media_gols_copa']
    Xte_b_raw = df_v5[mask_te_b][features_v5]

    imp = SimpleImputer(strategy='median')
    Xtr_b = imp.fit_transform(Xtr_b_raw)
    Xte_b = imp.transform(Xte_b_raw)

    for suf, cls in [('LR', LinearRegression()),
                     ('RF', RandomForestRegressor(n_estimators=100, random_state=42)),
                     ('XGB', XGBRegressor(n_estimators=100, random_state=42))]:
        if len(Xtr_a) > 0:
            cls.fit(Xtr_a, ytr_a)
            mae_cv_A[suf].append(mean_absolute_error(yte, cls.predict(Xte_a)))

        if len(Xtr_b) > 0:
            from sklearn.base import clone as skclone
            m = skclone(cls)
            m.fit(Xtr_b, ytr_b)
            mae_cv_B[suf].append(mean_absolute_error(
                df_v5[mask_te_b]['media_gols_copa'],
                m.predict(Xte_b)
            ))

nomes = {'LR': 'Reg. Linear', 'RF': 'Random Forest', 'XGB': 'XGBoost'}
print('Validação Cruzada — Modelo A vs B:')
print(f'{"Modelo":<25} {"A MAE":>8} {"A Std":>8} {"B MAE":>8} {"B Std":>8}')
print('-' * 60)
for suf in ['LR', 'RF', 'XGB']:
    a_mae = np.mean(mae_cv_A[suf]) if mae_cv_A[suf] else float('nan')
    a_std = np.std(mae_cv_A[suf])  if mae_cv_A[suf] else float('nan')
    b_mae = np.mean(mae_cv_B[suf]) if mae_cv_B[suf] else float('nan')
    b_std = np.std(mae_cv_B[suf])  if mae_cv_B[suf] else float('nan')
    print(f'{nomes[suf]:<25} {a_mae:>8.4f} {a_std:>8.4f} {b_mae:>8.4f} {b_std:>8.4f}')

## 6. Importância de Features — Modelo B

In [ ]:
from matplotlib.patches import Patch

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (nome, modelo) in zip(axes, [
    ('Random Forest B', modelos_B['Random Forest']),
    ('XGBoost B',       modelos_B['XGBoost'])
]):
    imp = pd.Series(modelo.feature_importances_, index=features_v5).sort_values()
    novas_v5 = ['media_gols_ultimas2_copas', 'fase_ultima_copa', 'valor_mercado_milhoes']
    colors = ['seagreen' if f in novas_v5 else 'steelblue' for f in imp.index]
    imp.plot(kind='barh', ax=ax, color=colors, edgecolor='black')
    ax.set_title(f'Importância de Features — {nome}')
    ax.set_xlabel('Importância')

legend = [
    Patch(color='seagreen',  label='Features novas v5'),
    Patch(color='steelblue', label='Features v4')
]
axes[0].legend(handles=legend, loc='lower right')
plt.tight_layout()
plt.savefig('../article/figures/importancia_features_v5.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Tabela Final e Conclusão

In [ ]:
print('TABELA COMPARATIVA FINAL:')
print(df_res.to_string(index=False))

df_res.to_csv('../article/tables/comparacao_modeloA_modeloB.csv', index=False)
print('\nTabela salva em article/tables/comparacao_modeloA_modeloB.csv')

## 8. Conclusão

**Critério de decisão:**
- Se Modelo B MAE < Modelo A MAE → features adicionais compensam a redução de amostras
- Se Modelo A MAE ≤ Modelo B MAE → mais dados superam mais features (parcimônia)

**Independente do resultado:** o experimento é válido e vai para o artigo como análise da troca entre riqueza de features e tamanho do dataset de treino (bias-variance tradeoff).

**Próximo passo:** Atualizar `04_previsao_2026` com o modelo vencedor.